# CME538 — Introduction to Data Science
## Week 2 | Lecture 2.3: Pandas III

This is the **third block of today's Pandas session**, following Pandas I and Pandas II.

### Topics
- Lambda functions
- Iterating through DataFrames
- Vectorization and performance
- Combining DataFrames with `concat()` and `merge()`

---

## Setup

We will continue using **Pandas** and **NumPy**. We also import `time` so that we can compare how long different approaches take.

In [ ]:
# TODO: Import Pandas and NumPy using their standard aliases.

import time



---

# 1. Lambda Functions

Earlier, we defined our own functions using `def`.

A **lambda function** is a small, anonymous function written in a single expression. It is useful when we need a simple function temporarily and do not need to give it a permanent name.

In [ ]:
# TODO: Define a function that raises a number to a power.



The same simple function can be written using `lambda`:

```text
lambda arguments: expression
```

The expression after the colon is automatically returned.

In [ ]:
# TODO: Rewrite raise_to_power as a lambda function.



### Connection to Pandas II

We already used a lambda function when filtering election-year groups:

```python
elections.groupby("Year").filter(
    lambda group: group["%"].max() < 45
)
```

The lambda is simply a compact version of a regular function.

In [ ]:
# These two functions express the same rule.

def keep_group(group):
    return group["%"].max() < 45

# Equivalent:
keep_group_lambda = lambda group: group["%"].max() < 45

> **Key idea:** Use `def` when a function is more substantial or will be reused. A `lambda` is convenient for a short function used in place.

---

# 2. Iterating Through DataFrames

There are several ways to perform a calculation for many DataFrame rows. For large datasets, the choice can have a major effect on computation time.

We will compare several approaches using the same task: calculate the distance from many geographic locations to Toronto.

<div style="display: flex; align-items: center; gap: 45px;">

<div style="width: 42%; font-family: inherit;">

<h3>The Task</h3>

<p>
Suppose we have many geographic positions represented by <code>latitude</code> and <code>longitude</code>.
</p>

<p>
For each position, we want to calculate its distance from Toronto.
</p>

<p>
We will use the <strong>Haversine distance</strong>, which accounts for the curvature of the Earth.
</p>

</div>

<div style="width: 58%; text-align: center;">
<img src="images/map.png" alt="Haversine distance task"">
<img src="images/haversine_function.png" alt="Haversine distance task">
</div>

</div>

### Create the example data

The slides use **100,000 random locations**. We use a fixed random seed so everyone generates the same locations.

In [ ]:
# TODO: Create 100,000 random latitude/longitude locations.
# Number of random geographic locations to create.

# Create a random number generator.
# The seed 0 makes the random values reproducible.

# Create a DataFrame of random latitude and longitude coordinates.

    # Generate latitudes between -90° and 90°.

    # Generate longitudes between -180° and 180°.


# Display the first 5 locations.



In [ ]:
# Coordinates used in this example.

toronto_lat = 43.651070
toronto_lon = -79.347015

### Haversine function

The details of the formula are not the focus here. We will use the **same function in every method** so that the comparison is about how we process the DataFrame.

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """Calculate great-circle distance in miles."""
    miles = 3959

    lat1, lon1, lat2, lon2 = map(
        np.deg2rad,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))
    return miles * c

## Method 1: Loop over row positions with `range()`

A natural first approach is to loop through row positions one at a time.

This works, but repeatedly accessing individual DataFrame values does not take advantage of Pandas' optimized array operations.

In [ ]:
def simple_for_loop_method(df):
    distances = []

    # TODO: Loop through each row position.

    result = df.copy()
    result["distance"] = distances
    return result



In [ ]:
# Run the method once and measure the elapsed time.
start = time.perf_counter()
result_range = simple_for_loop_method(locations)
elapsed_range = time.perf_counter() - start

print(f"range() loop: {elapsed_range:.3f} seconds")
result_range.head()

## Method 2: `.iterrows()`

`.iterrows()` iterates through a DataFrame one row at a time. Each iteration gives us:

- the row's **Index label**
- a **Series** containing that row

It is easy to read, but row-by-row iteration can be slow for large DataFrames.

In [ ]:
def iterrows_method(df):
    distances = []

    # TODO: Iterate through the DataFrame one row at a time.

    result = df.copy()
    result["distance"] = distances
    return result



In [ ]:
start = time.perf_counter()
result_iterrows = iterrows_method(locations)
elapsed_iterrows = time.perf_counter() - start

print(f"iterrows(): {elapsed_iterrows:.3f} seconds")

## Method 3: Convert rows to dictionaries

`to_dict(orient="records")` converts the DataFrame into a list of dictionaries, where each dictionary represents one row.

This still uses a Python loop, but it gives us another way to see that the same task can be implemented differently.

In [ ]:
def to_dict_method(df):
    distances = []

    # TODO: Loop through rows represented as dictionaries.

    result = df.copy()
    result["distance"] = distances
    return result



In [ ]:
start = time.perf_counter()
result_dict = to_dict_method(locations)
elapsed_dict = time.perf_counter() - start

print(f"to_dict(): {elapsed_dict:.3f} seconds")

## Method 4: Pandas `.apply()`

`.apply()` applies a function along an axis of a DataFrame.

Here we want the function to receive **one row at a time**, so we use `axis=1`.

Recall:

- `axis=0` → operate along the row Index / down rows
- `axis=1` → operate across columns / one row at a time

In [ ]:
def apply_method(df):
    result = df.copy()

    # TODO: Apply the Haversine calculation to each row.
    # axis=1 means the lambda receives one row at a time.

    return result



In [ ]:
start = time.perf_counter()
result_apply = apply_method(locations)
elapsed_apply = time.perf_counter() - start

print(f"apply(): {elapsed_apply:.3f} seconds")

## Method 5: Vectorization with Pandas Series

**Vectorization** means performing an operation on an entire array or Series rather than explicitly processing one value at a time in Python.

Our `haversine()` function uses NumPy operations, so it can accept entire Pandas Series for latitude and longitude.

In [ ]:
def vectorized_series_method(df):
    result = df.copy()

    # TODO: Calculate all distances at once using entire Series.

    return result



In [ ]:
start = time.perf_counter()
result_series = vectorized_series_method(locations)
elapsed_series = time.perf_counter() - start

print(f"Vectorized Pandas Series: {elapsed_series:.6f} seconds")

## Method 6: Vectorization with NumPy arrays

Pandas Series are built on array-based data structures. We can also pass the underlying values as NumPy arrays using `.to_numpy()`.

In [ ]:
def vectorized_numpy_method(df):
    result = df.copy()

    # TODO: Convert the coordinate Series to NumPy arrays.

    return result



In [ ]:
start = time.perf_counter()
result_numpy = vectorized_numpy_method(locations)
elapsed_numpy = time.perf_counter() - start

print(f"Vectorized NumPy arrays: {elapsed_numpy:.6f} seconds")

### Compare the methods

Exact timings depend on the computer and software environment, so focus on the **relative pattern**, not the specific numbers printed in the slides.

In [ ]:
timings = pd.Series({
    "range() loop": elapsed_range,
    "iterrows()": elapsed_iterrows,
    "to_dict()": elapsed_dict,
    "apply()": elapsed_apply,
    "Vectorized Series": elapsed_series,
    "Vectorized NumPy": elapsed_numpy
}, name="Seconds")

timings.sort_values()

> **Key lesson:** There are many ways to process DataFrame rows, but they can have very different performance. When possible, prefer clear **vectorized operations** over explicit Python row-by-row loops.

Performance is not the only goal: good code should also be **modular** and **easy to understand**.

---

# 3. Combining DataFrames

Real analyses often use data from multiple files or sources. We therefore need ways to combine DataFrames.

Two important Pandas tools are:

- `pd.concat()` — stack DataFrames by position
- `.merge()` — match rows using values in one or more columns

<div style="display: flex; align-items: center; gap: 45px;">

<div style="width: 40%; font-family: inherit;">

<h3>Concat or Merge?</h3>

<p>
The key question is whether values inside the DataFrames are needed to decide which rows belong together.
</p>

<p>
If we simply want to <strong>stack</strong> datasets, use <code>concat()</code>.
</p>

<p>
If we need to <strong>match</strong> rows using a shared key, use <code>merge()</code>.
</p>

</div>

<div style="width: 60%; text-align: center;">
<img src="images/combining_overview.png">


<div>

<div>

## 3.1 Concatenating DataFrames

Imagine that Uber trip data are stored in separate files for different months. Each file has the **same columns**, and each row represents a trip.

To analyze all months together, we want to stack the rows into one DataFrame.

We will use a small version of that structure so the example is self-contained.

In [ ]:
# Small stand-in for monthly Uber trip files.

april_data = pd.DataFrame({
    "Date/Time": ["4/1/2014 0:11", "4/1/2014 0:17"],
    "Lat": [40.7690, 40.7267],
    "Lon": [-73.9549, -74.0345],
    "Base": ["B02512", "B02512"]
})

may_data = pd.DataFrame({
    "Date/Time": ["5/1/2014 0:02", "5/1/2014 0:06"],
    "Lat": [40.7521, 40.6965],
    "Lon": [-73.9914, -73.9715],
    "Base": ["B02598", "B02598"]
})

april_data

### Stack rows: `axis=0`

For monthly files with the same columns, we want one month's rows underneath another month's rows.

In [ ]:
# TODO: Stack the April and May DataFrames vertically.
# axis=0 means stack along the row axis.



Notice that the original Index labels were preserved, so `0` and `1` appear more than once.

When the old row labels are no longer meaningful, `ignore_index=True` creates a fresh Index.

In [ ]:
# TODO: Concatenate again and create a new continuous Index.



### Stack columns: `axis=1`

`pd.concat()` can also place DataFrames side-by-side using `axis=1`.

This aligns rows using their **Index labels**.

In [ ]:
left_demo = pd.DataFrame({"Name": ["Ava", "Noah", "Mia"]})
right_demo = pd.DataFrame({"Score": [82, 91, 88]})

# TODO: Place the two DataFrames side-by-side.



> **Concat mental model:** `concat()` primarily stacks objects. It does not search a key column to decide which records represent the same entity.

---

## 3.2 Merging DataFrames

Use `.merge()` when values in the DataFrames determine which rows should be matched.

Suppose one DataFrame contains participant names and another contains test scores. Both contain a shared key: `participant_id`.

In [ ]:
# Participant information.

df1 = pd.DataFrame({
    "participant_id": ["1", "6", "33", "42", "65", "8", "20", "13", "14"],
    "first_name": [
        "Shoshanna", "Marianne", "Karl", "Brent", "John",
        "Marcus", "Bruce", "Judi", "Denzel"
    ],
    "last_name": [
        "Saxe", "Touchie", "Peterson", "Sleep", "Harrison",
        "Aurelius", "Wayne", "Dench", "Washington"
    ]
})

df1

In [ ]:
# Test scores.

df2 = pd.DataFrame({
    "participant_id": [
        "22", "98", "71", "33", "42", "65",
        "8", "20", "13", "14", "34", "54"
    ],
    "score": [80, 76, 72, 66, 77, 64, 59, 60, 62, 89, 67, 58]
})

df2

<div style="display: flex; align-items: center; gap: 45px;">

<div style="width: 42%; font-family: inherit;">

<h3>Four Common Join Types</h3>

<p>
The <code>how=</code> argument controls which keys are retained.
</p>

<p>
We will compare <strong>outer</strong>, <strong>left</strong>, <strong>right</strong>, and <strong>inner</strong> joins using <code>participant_id</code>.
</p>

</div>

<div style="width: 58%; text-align: center;">
<img src="images/joins.jpg" alt="Outer, left, right, and inner join diagrams" style="width: 80%;">
</div>

</div>

### Outer Join

An **outer join** keeps every key that appears in either DataFrame.

When a participant exists on only one side, the missing values from the other DataFrame appear as `NaN`.

In [ ]:
# TODO: Keep all participant IDs from both DataFrames.



### Left Join

A **left join** keeps every row from the **left** DataFrame (`df1`) and adds matching information from the right DataFrame (`df2`).

Participants in `df1` without a matching score are still retained.

In [ ]:
# TODO: Keep all participants from df1 and add matching scores from df2.



### Right Join

A **right join** keeps every row from the **right** DataFrame (`df2`) and adds matching information from the left DataFrame (`df1`).

Scores without matching participant information are still retained.

In [ ]:
# TODO: Keep all scores from df2 and add matching names from df1.



### Inner Join

An **inner join** keeps only keys that appear in **both** DataFrames.

Here, only participants who have both name information and a score remain.

In [ ]:
# TODO: Keep only participant IDs that appear in both DataFrames.



### Quick comparison

For this example:

- **outer** → all IDs from either table
- **left** → all IDs from `df1`
- **right** → all IDs from `df2`
- **inner** → only IDs shared by both tables

The word **left** or **right** refers to the DataFrame's position in the merge operation.

In [ ]:
# Compare the number of rows produced by each join.

pd.Series({
    "df1": len(df1),
    "df2": len(df2),
    "outer": len(df_outer_join),
    "left": len(df_left_join),
    "right": len(df_right_join),
    "inner": len(df_inner_join)
})

---

# Wrap-Up

In this third Pandas block, we added three ideas:

1. **Lambda functions** provide a compact way to define simple functions.
2. DataFrames can be processed row-by-row, but **vectorized operations** are often much faster.
3. DataFrames can be combined using:
   - `pd.concat()` when we want to **stack** data
   - `.merge()` when we want to **match records using a key**

Together with Pandas I and II, these tools form a practical foundation for selecting, transforming, grouping, and combining tabular data.